In [ ]:
import pathlib as pl
from pprint import pprint
import shutil

import jupyter_black
import numpy as np
import pywatershed as pws
import xarray as xr

jupyter_black.load()

In [ ]:
input_dir = pl.Path("../test_data/drb_2yr/output/")
domain_dir = pws.constants.__pywatershed_root__ / "data/drb_2yr"

t0 = np.datetime64("1979-01-01T00:00:00")
timestep = np.timedelta64(24, "h")
n_steps = 5

nb_output_dir = pl.Path("./08_restart_streamflow")
restart_dir = nb_output_dir / "restart"
if restart_dir.exists():
    shutil.rmtree(restart_dir)

In [ ]:
def get_control(init_time, end_time, restart_read: pl.Path = False):
    control = pws.Control.load_prms(
        domain_dir / "nhm.control", warn_unused_options=False
    )
    control.options["input_dir"] = input_dir
    control.options["restart_write"] = restart_dir
    control.options["restart_write_freq"] = "d"
    if restart_read:
        control.options["restart_read"] = restart_read
    del control.options["netcdf_output_dir"]
    del control.options["netcdf_output_var_names"]
    control.edit_end_time(end_time)
    control.edit_init_start_times(init_time)
    return control


control = get_control(t0, t0 + timestep)

In [ ]:
def get_params():
    return pws.parameters.PrmsParameters.load(domain_dir / "myparam.param")

In [ ]:
nhm_processes = [
    pws.PRMSGroundwater,
    pws.PRMSChannel,
]

In [ ]:
nhm = pws.Model(
    nhm_processes,
    control=control,
    parameters=get_params(),
)
nhm.run(finalize=True)

In [ ]:
pprint(sorted(restart_dir.glob("*.nc")))

In [ ]:
for ii in range(n_steps - 1):
    init_time = t0 + (ii + 1) * timestep
    nhm = pws.Model(
        nhm_processes,
        control=get_control(
            init_time, init_time + timestep, restart_read=restart_dir
        ),
        parameters=get_params(),
    )
    nhm.run(finalize=True)
    print(f"{ii=}: {init_time=}")
    pprint(sorted(restart_dir.glob("*.nc")))

In [ ]:
restart_dir = nb_output_dir / "restart_2"
if restart_dir.exists():
    shutil.rmtree(restart_dir)

nhm = pws.Model(
    nhm_processes,
    control=get_control(t0, t0 + timestep * n_steps),
    parameters=get_params(),
)
nhm.run(finalize=True)
pprint(sorted(restart_dir.glob("*.nc")))

In [ ]:
final_time_stamp = (t0 + n_steps * timestep).item().strftime("%Y-%m-%d")
for no_rs_file in sorted(restart_dir.glob(f"{final_time_stamp}*")):
    rs_file = nb_output_dir / f"restart/{no_rs_file.name}"
    print(no_rs_file)
    print(rs_file)
    no_rs_da = xr.load_dataarray(no_rs_file)
    rs_da = xr.load_dataarray(rs_file)
    # xr.testing.assert_allclose(no_rs_da, rs_da)
    xr.testing.assert_equal(no_rs_da, rs_da)
    print(no_rs_file.name, "passes")